# Deteção da Região Spike — SARS-CoV-2

Fluxo:
1. Leitura dos dois genomas FASTA
2. Alinhamento global **Needleman-Wunsch** (Biopython) → `seq_ref_aln` / `seq_var_aln`
3. Divergência em **janelas de 1 000 bp** sobre o alinhamento
4. Gráfico `grafico_spike` com mapa de genes e subdomínios da Spike

Genomas:
- **Referência**: NC_045512.2 (Wuhan-Hu-1)
- **Variante**: MZ359844.1 (amostra indiana 2021 — Delta/Kappa)

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from Bio.Align import PairwiseAligner

# ── Leitura de FASTA ──────────────────────────────────────────────────────────
def read_fasta(path):
    seq, header = [], ""
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line.startswith(">"):
                header = line[1:]
            else:
                seq.append(line.upper())
    return header, "".join(seq)

# ── Divergência por janelas sobre sequências alinhadas ────────────────────────
def windowed_divergence(seq_ref_aln, seq_var_aln, window=1000):
    """
    Percorre as sequências alinhadas (mesmo comprimento, com gaps '-')
    em janelas de `window` colunas de alinhamento.
    Devolve:
      positions_ref  — posição central em coordenadas da referência
      divergence     — nº de colunas divergentes (mismatch ou gap) por janela
    """
    aln_len = len(seq_ref_aln)

    # Mapa: índice de coluna → coordenada na referência (ignora gaps na ref)
    ref_coord = 0
    ref_pos_map = []
    for c in seq_ref_aln:
        ref_pos_map.append(ref_coord)
        if c != '-':
            ref_coord += 1

    positions_ref, divergence = [], []

    for aln_start in range(0, aln_len - window + 1, window):
        aln_end = aln_start + window
        chunk_ref = seq_ref_aln[aln_start:aln_end]
        chunk_var = seq_var_aln[aln_start:aln_end]

        div = sum(1 for r, v in zip(chunk_ref, chunk_var) if r != v)
        ref_mid = ref_pos_map[aln_start + window // 2]

        positions_ref.append(ref_mid)
        divergence.append(div)

    return positions_ref, divergence

## 1. Carregamento dos genomas

In [ ]:
REF_PATH = "../SARS-CoV-2_ref.fasta"
VAR_PATH = "../SARS-CoV-2_variant.fasta"

hdr_ref, seq_ref = read_fasta(REF_PATH)
hdr_var, seq_var = read_fasta(VAR_PATH)

print(f"Referência : {hdr_ref[:80]}")
print(f"  Comprimento: {len(seq_ref):,} nt")
print()
print(f"Variante   : {hdr_var[:80]}")
print(f"  Comprimento: {len(seq_var):,} nt")

## 2. Alinhamento global Needleman-Wunsch

O `PairwiseAligner` do Biopython corre NW em C.
Para ~30 kb cada, demora alguns segundos e pode usar até ~3.6 GB de RAM.
As sequências alinhadas resultantes têm o mesmo comprimento e contêm `-` nos gaps.

In [ ]:
aligner = PairwiseAligner()
aligner.mode             = 'global'   # Needleman-Wunsch
aligner.match_score      =  2
aligner.mismatch_score   = -1
aligner.open_gap_score   = -2
aligner.extend_gap_score = -0.5

print("A alinhar genomas (Needleman-Wunsch)...")
t0 = time.time()
alignment = next(iter(aligner.align(seq_ref, seq_var)))
elapsed = time.time() - t0

seq_ref_aln = str(alignment[0])
seq_var_aln = str(alignment[1])

print(f"  Concluído em {elapsed:.1f}s")
print(f"  Score                     : {alignment.score:.0f}")
print(f"  Comprimento do alinhamento: {len(seq_ref_aln):,} colunas")
print(f"  Gaps na referência        : {seq_ref_aln.count('-'):,}")
print(f"  Gaps na variante          : {seq_var_aln.count('-'):,}")

## 3. Divergência por janelas de 1 000 bp

In [ ]:
WINDOW = 1000

positions_ref, divergence = windowed_divergence(seq_ref_aln, seq_var_aln, window=WINDOW)

divergence_per_bp = [d / WINDOW for d in divergence]
cumulative_norm   = list(np.cumsum(divergence_per_bp))

print(f"Janelas analisadas      : {len(positions_ref)}")
print(f"Divergência total       : {sum(divergence):,} colunas")
print(f"Divergência média / bp  : {np.mean(divergence_per_bp):.4f}")
print()
print("Top 5 janelas mais divergentes:")
ranked = sorted(zip(divergence_per_bp, positions_ref), reverse=True)
for dpb, pos in ranked[:5]:
    print(f"  Posição ref ~{pos:>6,} nt  →  {dpb:.4f} div/bp  ({int(dpb*WINDOW)} colunas divergentes)")

## 4. Gráfico — `grafico_spike`

In [ ]:
# ── Anotações biológicas (coordenadas NC_045512.2) ────────────────────────────
GENES = [
    ("ORF1ab", 266,   21555, "#4e79a7"),
    ("S",      21563, 25384, "#e15759"),
    ("ORF3a",  25393, 26220, "#f28e2b"),
    ("E",      26245, 26472, "#76b7b2"),
    ("M",      26523, 27191, "#59a14f"),
    ("ORF6",   27202, 27387, "#edc948"),
    ("ORF7a",  27394, 27759, "#b07aa1"),
    ("ORF7b",  27756, 27887, "#ff9da7"),
    ("ORF8",   27894, 28259, "#9c755f"),
    ("N",      28274, 29533, "#bab0ac"),
    ("ORF10",  29558, 29674, "#d4a6c8"),
]

SPIKE_DOMAINS = [
    ("NTD", 21563, 22599, "#ffb3b3"),
    ("RBD", 22517, 23183, "#ff4444"),
    ("S2",  23623, 25384, "#ff8080"),
]
FURINA_POS = 23603  # sítio de clivagem por furina (P681R em Delta)

# ── Figura com 3 painéis ──────────────────────────────────────────────────────
fig, (ax1, ax2, ax3) = plt.subplots(
    3, 1, figsize=(16, 11), sharex=True,
    gridspec_kw={"height_ratios": [1.8, 1.6, 0.55]}
)
fig.suptitle(
    "Divergência SARS-CoV-2: Wuhan-Hu-1 vs. Variante Indiana MZ359844.1 (Delta/Kappa)\n"
    "Alinhamento Needleman-Wunsch · Janelas de 1 000 bp · Coordenadas NC_045512.2",
    fontsize=13, fontweight="bold"
)

x          = positions_ref
genome_len = len(seq_ref)

# ── Painel 1: divergência cumulativa normalizada ──────────────────────────────
ax1.plot(x, cumulative_norm, color="#1f77b4", linewidth=2,
         label="Divergência cumulativa (por bp)")
ax1.axvspan(21563, 25384, alpha=0.12, color="#e15759", zorder=0, label="Região Spike")
ax1.set_ylabel("Divergência\ncumulativa / bp", fontsize=10)
ax1.legend(fontsize=9, loc="upper left")
ax1.grid(axis="y", linestyle="--", alpha=0.35)

max_idx = int(np.argmax(divergence_per_bp))
ax1.annotate(
    f"Pico: ~{x[max_idx]:,} nt\n({divergence_per_bp[max_idx]:.3f} div/bp)",
    xy=(x[max_idx], cumulative_norm[max_idx]),
    xytext=(x[max_idx] - 5000, cumulative_norm[max_idx] * 0.80),
    arrowprops=dict(arrowstyle="->", color="#333333"),
    fontsize=8.5, color="#333333",
    bbox=dict(boxstyle="round,pad=0.3", fc="white", alpha=0.8)
)

# ── Painel 2: barras por janela + subdomínios Spike ───────────────────────────
for name, s, e, color in SPIKE_DOMAINS:
    ax2.axvspan(s, e, alpha=0.22, color=color, zorder=0)
    ax2.text((s + e) / 2, max(divergence_per_bp) * 1.03, name,
             ha="center", va="bottom", fontsize=7.5,
             color="#6b0000", fontweight="bold")

ax2.axvline(FURINA_POS, color="#8b0000", linewidth=1.2, linestyle="--", zorder=3)
ax2.text(FURINA_POS + 80, max(divergence_per_bp) * 0.97, "Furina\nP681R",
         fontsize=6.5, color="#8b0000", va="top")

bar_colors = ["#e15759" if 21563 <= p <= 25384 else "#aec7e8" for p in x]
ax2.bar(x, divergence_per_bp, width=WINDOW * 0.85, color=bar_colors,
        align="center", zorder=2)
ax2.set_ylabel("Divergência\npor bp / janela", fontsize=10)
ax2.grid(axis="y", linestyle="--", alpha=0.35)

spike_patch = mpatches.Patch(color="#e15759", alpha=0.8, label="Spike (S)")
other_patch = mpatches.Patch(color="#aec7e8", label="Restante genoma")
furina_line = plt.Line2D([0], [0], color="#8b0000", linestyle="--",
                         label="Sítio furina (~23 603 nt)")
ax2.legend(handles=[spike_patch, other_patch, furina_line],
           fontsize=8.5, loc="upper left")

# ── Painel 3: mapa de genes ───────────────────────────────────────────────────
ax3.set_ylim(0, 1)
ax3.set_yticks([])
ax3.set_ylabel("Genes", fontsize=9)
ax3.set_xlabel("Posição no genoma de referência (nt)", fontsize=11)

for name, s, e, color in GENES:
    ax3.barh(0.5, e - s, left=s, height=0.55, color=color,
             align="center", edgecolor="white", linewidth=0.4)
    if e - s > 600:
        ax3.text((s + e) / 2, 0.5, name,
                 ha="center", va="center", fontsize=7,
                 color="white", fontweight="bold")

ax3.set_xlim(0, genome_len)
ax3.grid(False)

plt.tight_layout(h_pad=0.6)
plt.savefig("../grafico_spike.png", dpi=150, bbox_inches="tight")
plt.show()
print("Gráfico guardado em ../grafico_spike.png")